# Synthetic Probe Test — Evaluation on Real Extraction Data

Load trained probes saved by `synthetic_probe_analysis.ipynb` and evaluate them
on real extraction data.  Produces publication-ready figures and summary tables
for every combination of (train dataset, test dataset).

**Prerequisites**: Run `synthetic_probe_analysis.ipynb` for each
(dataset, judge_model) combination before running this notebook.

---

**Four evaluation settings** (for `DATASETS = ['pond', 'nfix']`):
1. pond trained → pond tested (within-domain)
2. nfix trained → nfix tested (within-domain)
3. pond trained → nfix tested (cross-domain)
4. nfix trained → pond tested (cross-domain)

**Per-combination outputs**:
- Calibration curve: NTP baseline vs. EM-rescaled probe predictions for each judge model
- Summary table: recovery rate, hallucination rate, ECE at threshold = 0.5
- Threshold sweep: recovery rate vs. hallucination rate as threshold varies 0.5 → 0.95

**Label definition** (`LABEL_MODE = 'union'`): a real extraction is positive
if it matches a ground-truth row **or** the majority-vote frontier judge says valid.

In [ ]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

from analysis.loaders import (
    load_activations, load_combined_judgements,
    load_extraction, load_ground_truth, load_trained_probe,
    cached_match,
)
from scholarlm.utils.calibration import reliability_diagram_data, rescale_probabilities_em
from scholarlm.utils.unit_conversion import apply_unit_conversion
from experiments.run_extraction import load_dataset_config
import paths

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "text.usetex": False,
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "legend.title_fontsize": 9,
    "axes.linewidth": 0.6,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.major.size": 3, "ytick.major.size": 3,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "lines.linewidth": 1.2, "lines.markersize": 4,
    "legend.frameon": False,
    "figure.dpi": 150, "savefig.dpi": 300,
    "savefig.format": "pdf", "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

FIGURES_DIR = "../figures/synthetic_probe/"
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
EXTRACTION_MODEL = 'gemma-3-27b'
DATASETS         = ['pond', 'nfix']
JUDGE_MODELS     = ['llama-3.1-8b', 'mistral-7b', 'gemma-2-9b']

# Extraction date per test dataset
EXTRACTION_DATES = {
    'pond': 'FILL_IN',
    'nfix': '2026_04_25',
}

# Judge date for real activations: {test_dataset: {judge_model: date_str | None}}
# None → auto-detect latest
JUDGE_DATES_REAL = {
    'pond': {
        'llama-3.1-8b': None,
        'mistral-7b':   None,
        'gemma-2-9b':   None,
    },
    'nfix': {
        'llama-3.1-8b': None,
        'mistral-7b':   None,
        'gemma-2-9b':   '2026_04_28',
    },
}

THRESHOLD_SWEEP = np.linspace(0.5, 0.95, 15)  # thresholds for operating-curve plot
EDGE_THRESHOLD  = 1 / 3                        # minimum fuzzy weight to count as a match

# Visual style
JUDGE_COLORS = {
    'llama-3.1-8b': '#3984e0',
    'mistral-7b':   '#2eb07b',
    'gemma-2-9b':   '#d946a6',
}

## Matching Configuration

In [ ]:
SUPERSCRIPT_MAP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺",
                                "0123456789-+")
SUBSCRIPT_MAP   = str.maketrans("₀₁₂₃₄₅₆₇₈₉₋₊",
                                "0123456789-+")
SCRIPT_MAP = {**SUPERSCRIPT_MAP, **SUBSCRIPT_MAP}
LATEX_SCRIPT_RE = re.compile(r"[\^_]\{([^}]*)\}|[\^_]([+-]?\d+)")
ELEMENT_DASH_RE = re.compile(r"(mol|g) (N|C)")
TIME_UNIT_RE    = re.compile(r"\bday\b|\bhr\b")
TIME_UNIT_SUB   = {"day": "d", "hr": "h"}


def nfix_clean_unit(s):
    if not isinstance(s, str):
        return s
    s = s.translate(SCRIPT_MAP)
    s = LATEX_SCRIPT_RE.sub(lambda m: m.group(1) if m.group(1) is not None else m.group(2), s)
    s = ELEMENT_DASH_RE.sub(r"\1-\2", s)
    s = TIME_UNIT_RE.sub(lambda m: TIME_UNIT_SUB[m.group()], s)
    return s


def get_matching_config(dataset):
    """Return (strict_matching, fuzzy_matching) dicts for a dataset."""
    if dataset == 'pond':
        strict = {'document_id': 'document_id', 'attribute': 'attribute',
                  'value': 'converted_value', 'units': 'units'}
        fuzzy  = {'name': 'name', 'location': 'location', 'ecosystem': 'ecosystem'}
    elif dataset == 'nfix':
        strict = {'document_id': 'document_id', 'attribute': 'attribute',
                  'value': 'converted_value', 'units': 'units'}
        fuzzy  = {'name': 'name', 'site_type': 'site_type'}
    else:
        raise ValueError(f'Unknown dataset: {dataset}')
    return strict, fuzzy

## Load and Cache Test Data

For each test dataset: load extraction results, combined judgements, ground truth,
compute union labels (matching OR frontier judge), and run `cached_match`.

In [ ]:
test_data = {}  # test_data[dataset] = dict with pre-loaded data

for ds in DATASETS:
    print(f'Loading test data for {ds}...')
    config       = load_dataset_config(ds)
    records      = load_extraction(ds, EXTRACTION_MODEL, EXTRACTION_DATES[ds])
    ext_df       = pd.DataFrame(records)
    ext_df       = apply_unit_conversion(ext_df, config.unit_conversion_table)

    if ds == 'nfix':
        ext_df['attribute'] = ext_df['attribute'].map({
            'nfix_rate_areal': 'nfix_rate', 'nfix_rate_volumetric': 'nfix_rate',
            'nfix_rate_mass': 'nfix_rate', 'nfix_rate': 'nfix_rate',
        })
        ext_df['units'] = ext_df['units'].apply(nfix_clean_unit)

    real_df      = pd.DataFrame(load_combined_judgements(ds, EXTRACTION_MODEL, EXTRACTION_DATES[ds]))
    gt_df        = load_ground_truth(config)

    strict, fuzzy = get_matching_config(ds)
    cache_path    = paths.extraction(ds, EXTRACTION_MODEL, EXTRACTION_DATES[ds]) / 'match_cache.pkl'
    matching, edges, edge_weights = cached_match(
        gt_df, ext_df,
        strict_matching=strict,
        fuzzy_matching=fuzzy,
        fuzzy_threshold=0.0,
        cache_path=cache_path,
    )

    # Union labels: positive if GT-matched OR frontier judge says valid
    ex_edge_exists = np.zeros(len(ext_df), dtype=bool)
    for (gt_idx, ex_idx), w in zip(edges, edge_weights):
        if w > EDGE_THRESHOLD:
            ex_edge_exists[int(ex_idx)] = True
    jlabels     = real_df['judgement_combined'].to_numpy(dtype=bool)
    real_labels = jlabels | ex_edge_exists

    test_data[ds] = {
        'extraction_df':  ext_df,
        'real_df':        real_df,
        'ground_truth_df': gt_df,
        'real_labels':    real_labels,
        'ex_edge_exists': ex_edge_exists,
        'edges':          edges,
        'edge_weights':   edge_weights,
    }

    pos = real_labels.sum()
    print(f'  {ds}: {len(ext_df)} extractions, {len(gt_df)} GT rows, '
          f'pos={pos} ({pos/len(real_labels):.1%})')
    print()

## Load Trained Probes

In [ ]:
# probe_cache[train_dataset][judge_model] = probe_data dict
probe_cache = {}
for ds in DATASETS:
    probe_cache[ds] = {}
    for jm in JUDGE_MODELS:
        probe_cache[ds][jm] = load_trained_probe(ds, jm)
        top = probe_cache[ds][jm]['top_k_heads']
        print(f'  {ds} / {jm}: top heads={top}  pi_tr={probe_cache[ds][jm]["train_prevalence"]:.3f}')

## Analysis Helpers

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import brier_score_loss


def _probe_metrics(probs, y_true, threshold=0.5):
    """Compute metrics at a fixed threshold. Returns dict."""
    probs   = np.asarray(probs)
    y_true  = np.asarray(y_true, dtype=bool)
    preds   = probs > threshold
    tp  = int(( preds &  y_true).sum())
    tn  = int((~preds & ~y_true).sum())
    fp  = int(( preds & ~y_true).sum())
    fn  = int((~preds &  y_true).sum())
    n   = len(y_true)
    acc   = (tp + tn) / n
    prec  = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    rec   = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else float('nan')
    auroc = roc_auc_score(y_true, probs) if y_true.sum() > 0 and (~y_true).sum() > 0 else float('nan')
    ece   = reliability_diagram_data(probs, y_true)['ece']
    bs    = float(brier_score_loss(y_true, probs))
    p_pos = float(y_true.mean())
    bss   = 1.0 - bs / (p_pos * (1 - p_pos)) if p_pos not in (0.0, 1.0) else float('nan')
    return dict(acc=acc, prec=prec, rec=rec, f1=f1, auroc=auroc,
                ece=ece, bs=bs, bss=bss, n=n)


def compute_rates(probs, threshold, real_test_labels, test_edges, n_gt):
    """Recovery rate and hallucination rate at a given threshold.

    At threshold t:
      accepted      = probs > t (accepted extractions)
      recovery      = fraction of GT rows matched by at least one accepted extraction
      hallucination = fraction of accepted extractions that are invalid
                      (not GT-matched and not judged valid by frontier judge)

    Parameters
    ----------
    probs            : 1-D array aligned with the test set
    threshold        : float
    real_test_labels : bool array aligned with the test set (union labels)
    test_edges       : list of (gt_idx, test_pos) pairs (pre-filtered to test set)
    n_gt             : total number of GT rows
    """
    accepted = np.asarray(probs) > threshold
    n_accepted = accepted.sum()
    if n_accepted == 0:
        return float('nan'), float('nan')
    gt_covered = np.zeros(n_gt, dtype=bool)
    for gt_idx, test_pos in test_edges:
        if accepted[test_pos]:
            gt_covered[gt_idx] = True
    recovery      = float(gt_covered.mean())
    hallucination = float(1 - real_test_labels[accepted].mean())
    return recovery, hallucination

In [ ]:
def analyze_combination(train_dataset, test_dataset):
    """Produce all outputs for one (train_dataset → test_dataset) combination."""
    label = f'{train_dataset} \u2192 {test_dataset}'
    print(f'\n{"="*72}')
    print(f'  Probe trained on: {train_dataset}    Tested on: {test_dataset}')
    print(f'{"="*72}')

    td = test_data[test_dataset]
    real_df         = td['real_df']
    ground_truth_df = td['ground_truth_df']
    real_labels_all = td['real_labels']
    edges           = td['edges']
    edge_weights    = td['edge_weights']
    n_gt            = len(ground_truth_df)

    # Union of syn_document_ids across all judge models for train_dataset
    syn_doc_ids = set()
    for jm in JUDGE_MODELS:
        syn_doc_ids.update(probe_cache[train_dataset][jm]['syn_document_ids'])

    # Filter: exclude test records from papers seen during synthetic training
    real_test_mask   = ~real_df['document_id'].isin(syn_doc_ids)
    real_test_idx    = np.where(real_test_mask.to_numpy())[0]
    real_test_mids   = real_df['measurement_id'].iloc[real_test_idx].tolist()
    real_test_labels = real_labels_all[real_test_idx]
    n_test = len(real_test_idx)
    n_pos  = real_test_labels.sum()

    print(f'  Test records   : {n_test}  pos={n_pos} ({n_pos/n_test:.1%})')
    print(f'  Excluded       : {(~real_test_mask).sum()} records '
          f'({real_df.loc[~real_test_mask, "document_id"].nunique()} papers from syn set)')

    # Precompute edges restricted to the test set (for threshold sweep)
    ex_to_test_pos = {int(real_test_idx[i]): i for i in range(n_test)}
    test_edges = [
        (int(gt_idx), ex_to_test_pos[int(ex_idx)])
        for (gt_idx, ex_idx), w in zip(edges, edge_weights)
        if int(ex_idx) in ex_to_test_pos and w > EDGE_THRESHOLD
    ]

    # ── Compute probabilities for each judge model ──────────────────────────
    ntp_probs_by_judge   = {}  # NTP baseline
    probe_probs_by_judge = {}  # EM-rescaled probe

    for jm in JUDGE_MODELS:
        pd_data = probe_cache[train_dataset][jm]
        top_k   = pd_data['top_k_heads']

        jdate = JUDGE_DATES_REAL.get(test_dataset, {}).get(jm, None)
        real_activations = load_activations(
            test_dataset, EXTRACTION_MODEL, EXTRACTION_DATES[test_dataset], jm, jdate
        )

        _all_real = {
            str(mid): np.array(real_activations[str(mid)], dtype=np.float32)
            for mid in real_test_mids
        }
        X_real = np.concatenate(
            [
                np.stack([_all_real[str(mid)][l, h, :] for mid in real_test_mids], axis=0)
                for l, h in top_k
            ],
            axis=1,
        )

        raw_probs = pd_data['probe'].predict_proba(X_real)[:, 1]
        cal_probs, pi_te_hat = rescale_probabilities_em(
            raw_probs, pi_tr=pd_data['train_prevalence']
        )
        print(f'  {jm}: EM pi_te_hat={pi_te_hat:.3f} (pi_tr={pd_data["train_prevalence"]:.3f})')

        ntp_probs_by_judge[jm]   = real_df[f'judgement_p_true_{jm}'].iloc[real_test_idx].to_numpy()
        probe_probs_by_judge[jm] = cal_probs

    # ── Figure 1: Calibration curves ────────────────────────────────────────
    fig_cal, ax_cal = plt.subplots(figsize=(3.5, 3.5))
    ax_cal.plot([0, 1], [0, 1], 'k--', lw=1.0, label='Perfect')

    for jm in JUDGE_MODELS:
        color = JUDGE_COLORS[jm]
        for probs, linestyle, tag in [
            (ntp_probs_by_judge[jm],   '--', 'NTP'),
            (probe_probs_by_judge[jm], '-',  'Probe'),
        ]:
            d     = reliability_diagram_data(probs, real_test_labels)
            valid = ~np.isnan(d['bin_accuracy'])
            ax_cal.plot(
                d['bin_centers'][valid], d['bin_accuracy'][valid],
                linestyle, color=color, lw=1.4, ms=3.5,
                label=f'{jm} {tag} (ECE={d["ece"]:.3f})',
            )

    ax_cal.set_xlim(0, 1); ax_cal.set_ylim(0, 1)
    ax_cal.set_xlabel('Mean predicted probability')
    ax_cal.set_ylabel('Observed frequency')
    ax_cal.set_title(f'Calibration — {label}')
    ax_cal.legend(fontsize=6, loc='upper left')
    ax_cal.grid(alpha=0.2)
    fig_cal.tight_layout()
    fig_cal.savefig(
        FIGURES_DIR + f'synprobe_test_cal_{train_dataset}_to_{test_dataset}.pdf',
        bbox_inches='tight',
    )
    plt.show()

    # ── Table: metrics at threshold = 0.5 ───────────────────────────────────
    rows = []
    for jm in JUDGE_MODELS:
        for probs, kind in [
            (ntp_probs_by_judge[jm],   'NTP baseline'),
            (probe_probs_by_judge[jm], 'Probe (EM-cal)'),
        ]:
            rec, hall = compute_rates(
                probs, 0.5, real_test_labels, test_edges, n_gt
            )
            ece = reliability_diagram_data(probs, real_test_labels)['ece']
            rows.append({
                'Judge model': jm,
                'Type':       kind,
                'Recovery':   rec,
                'Hallucination': hall,
                'ECE':        ece,
            })

    summary_df = pd.DataFrame(rows)
    print(f'\nSummary table ({label}, threshold=0.5):')
    print(summary_df.to_string(index=False, float_format='{:.3f}'.format))

    # ── Figure 2: Threshold sweep ────────────────────────────────────────────
    n_judges = len(JUDGE_MODELS)
    fig_sweep, axes_sweep = plt.subplots(1, n_judges, figsize=(3.5 * n_judges, 3.0),
                                          sharey=True)
    if n_judges == 1:
        axes_sweep = [axes_sweep]

    cmap  = cm.viridis
    norm  = mcolors.Normalize(vmin=THRESHOLD_SWEEP.min(), vmax=THRESHOLD_SWEEP.max())
    sm    = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    for ax, jm in zip(axes_sweep, JUDGE_MODELS):
        probs = probe_probs_by_judge[jm]
        sweep_rec  = []
        sweep_hall = []
        for t in THRESHOLD_SWEEP:
            r, h = compute_rates(probs, t, real_test_labels, test_edges, n_gt)
            sweep_rec.append(r)
            sweep_hall.append(h)

        sweep_rec  = np.array(sweep_rec)
        sweep_hall = np.array(sweep_hall)
        valid      = ~(np.isnan(sweep_rec) | np.isnan(sweep_hall))

        # Scatter colored by threshold
        sc = ax.scatter(
            sweep_hall[valid], sweep_rec[valid],
            c=THRESHOLD_SWEEP[valid], cmap=cmap, norm=norm,
            s=25, zorder=3,
        )
        ax.plot(sweep_hall[valid], sweep_rec[valid],
                '-', color='#888888', lw=0.8, zorder=2)

        # Mark threshold = 0.5
        idx0 = np.argmin(np.abs(THRESHOLD_SWEEP - 0.5))
        if valid[idx0]:
            ax.scatter([sweep_hall[idx0]], [sweep_rec[idx0]],
                       s=60, c='none', edgecolors='k', linewidths=1.0, zorder=4)

        ax.set_xlabel('Hallucination rate')
        if ax is axes_sweep[0]:
            ax.set_ylabel('Recovery rate')
        ax.set_title(jm, fontsize=8)
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.grid(alpha=0.2)

    fig_sweep.colorbar(sm, ax=axes_sweep[-1], label='Threshold', fraction=0.046, pad=0.04)
    fig_sweep.suptitle(f'Threshold sweep — {label}', fontsize=9)
    fig_sweep.tight_layout()
    fig_sweep.savefig(
        FIGURES_DIR + f'synprobe_test_sweep_{train_dataset}_to_{test_dataset}.pdf',
        bbox_inches='tight',
    )
    plt.show()

    return summary_df

---
## Within-Domain Evaluation

Probes trained and tested on the same dataset (no domain shift).

### pond → pond

In [ ]:
summary_pond_pond = analyze_combination('pond', 'pond')

### nfix → nfix

In [ ]:
summary_nfix_nfix = analyze_combination('nfix', 'nfix')

---
## Cross-Domain Evaluation

Probes trained on one dataset and tested on the other.

### pond → nfix

In [ ]:
summary_pond_nfix = analyze_combination('pond', 'nfix')

### nfix → pond

In [ ]:
summary_nfix_pond = analyze_combination('nfix', 'pond')